# Project 2: screening general classification datasets

**Purpose.** Find a Kaggle classification dataset that fits the Project 2 brief (Version 2, with data), outside the semiconductor domain. Every candidate goes through the same checks, and the notebook ends with a ranking.

This notebook is self-contained. It doesn't read or depend on anything else in the repository.

## What the brief asks, and the check that tests each one
| Brief | Check |
|---|---|
| Find a high-usability dataset | Not readable from code; check each Kaggle page (links in the candidate table) |
| Avoid large datasets | Rows (more than 100,000 = flagged) |
| Avoid missing data | Missing cells, counting `?`, `unknown` and blanks as missing |
| Avoid data cleaning | Duplicate rows, duplicates with conflicting labels |
| Avoid severe imbalance | Positive rate, and positives in a 20% validation split |
| If prediction is almost perfect, delete features | Untuned CV AUC; AUC after dropping the strongest feature |
| Evaluation loss deviates from misclassification, from the context | A stated cost ratio per dataset; how far the cost-minimising threshold moves from 0.5 and what it saves |
| Glassbox and blackbox, each with L2 and a surrogate loss | Out-of-fold AUC of ridge-on-0/1 vs logistic regression, and boosting with squared error vs log loss |
| Compare on validation AUC and decide | The blackbox − glassbox gap, and its spread across folds |
| Calibration | Expected calibration error (ECE) of the boosted trees |
| Tree SHAP, local and global | Tree SHAP runs; share of attribution carried by the top feature |

All checks use an 80% development split (a 20% test split is held out before any label is used), subsampled to 30,000 rows where larger. The models are untuned, so these are feasibility signals, not results.

## 0. Setup

In [1]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import kagglehub
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
MAX_ROWS = 30_000
MISSING_TOKENS = {"?", "unknown", "Unknown", "NA", "N/A", ""}
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)
print("pandas", pd.__version__)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas 3.0.5


## 1. Candidates
Each row gives the target, the positive class, the columns removed before modelling (IDs, and any column known only after the outcome), and the cost ratio assumed for the screening. The cost ratio is the cost of a missed positive in units of one false alarm; it is our assumption, chosen from the context of each dataset.

In [2]:
C = pd.DataFrame([
    # slug, file pattern, target, positive, drop, cost ratio (FN:FP), what a positive means, why the costs differ
    ("laotse/credit-risk-dataset", r"\.csv$", "loan_status", 1, [], 5,
     "Loan defaults", "A default loses principal; rejecting a good borrower loses interest"),
    ("uciml/default-of-credit-card-clients-dataset", r"\.csv$", "default.payment.next.month", 1, ["ID"], 5,
     "Card holder defaults next month", "A missed defaulter is written off; a false alarm cuts a good customer's limit"),
    ("imakash3011/online-shoppers-purchasing-intention-dataset", r"\.csv$", "Revenue", True, [], 5,
     "Session ends in a purchase", "A missed buyer is a lost sale; a discount to a non-buyer costs a little"),
    ("henriqueyamahata/bank-marketing", r"bank-additional-full", "y", "yes", ["duration"], 10,
     "Client subscribes to a term deposit", "A missed subscriber loses a deposit; a wasted call is cheap"),
    ("gauravtopre/bank-customer-churn-dataset", r"\.csv$", "churn", 1, ["customer_id"], 5,
     "Customer leaves the bank", "A lost customer costs lifetime value; a retention offer is cheap"),
    ("blastchar/telco-customer-churn", r"\.csv$", "Churn", "Yes", ["customerID"], 5,
     "Customer cancels service", "As above"),
    ("uciml/adult-census-income", r"\.csv$", "income", ">50K", [], 3,
     "Income above $50K", "A missed high earner is a lost premium-product sale"),
    ("sulianova/cardiovascular-disease-dataset", r"\.csv$", "cardio", 1, ["id"], 5,
     "Has cardiovascular disease", "A missed patient goes untreated; a false alarm costs a follow-up test"),
    ("jessemostipak/hotel-booking-demand", r"\.csv$", "is_canceled", 1, ["reservation_status", "reservation_status_date", "agent", "company"], 3,
     "Booking is cancelled", "An unexpected empty room costs a night's revenue; over-booking costs a relocation"),
    ("pavansubhasht/ibm-hr-analytics-attrition-dataset", r"\.csv$", "Attrition", "Yes", ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"], 5,
     "Employee leaves", "A lost employee costs rehiring; a retention talk is cheap"),
    ("kabure/german-credit-data-with-risk", r"\.csv$", "Risk", "bad", ["Unnamed: 0"], 5,
     "Bad credit risk", "Official cost matrix of the original data: 5 vs 1"),
    ("fedesoriano/stroke-prediction-dataset", r"\.csv$", "stroke", 1, ["id"], 10,
     "Has a stroke", "A missed stroke risk is severe; a false alarm costs a check-up"),
    ("fedesoriano/heart-failure-prediction", r"\.csv$", "HeartDisease", 1, [], 10,
     "Has heart disease", "As above"),
    ("uciml/pima-indians-diabetes-database", r"\.csv$", "Outcome", 1, [], 5,
     "Has diabetes", "A missed diagnosis goes untreated; a false alarm costs a test"),
    ("uciml/red-wine-quality-cortez-et-al-2009", r"\.csv$", "quality", ">=7", [], 3,
     "Wine rated 7 or higher", "A good wine sold as ordinary loses margin"),
    ("uom190346a/water-quality-and-potability", r"\.csv$", "Potability", 1, [], 5,
     "Water is potable", "Context-dependent"),
    ("uciml/faulty-steel-plates", r"\.csv$", "Bumps", 1, ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Other_Faults"], 10,
     "Plate has a Bumps fault (vs the six other fault types)", "A faulty plate shipped triggers a customer claim; a false alarm costs a re-inspection"),
], columns=["slug", "pattern", "target", "positive", "drop", "cost_ratio", "positive_means", "cost_story"])
C["kaggle_page"] = "https://www.kaggle.com/datasets/" + C["slug"]
display(C[["slug", "target", "positive_means", "cost_ratio", "cost_story"]])

,slug,target,positive_means,cost_ratio,cost_story
0,laotse/credit-risk-dataset,loan_status,Loan defaults,5,A default loses principal; rejecting a good borrower los...
1,uciml/default-of-credit-card-clients-dataset,default.payment.next.month,Card holder defaults next month,5,A missed defaulter is written off; a false alarm cuts a ...
2,imakash3011/online-shoppers-purchasing-intention-dataset,Revenue,Session ends in a purchase,5,A missed buyer is a lost sale; a discount to a non-buyer...
3,henriqueyamahata/bank-marketing,y,Client subscribes to a term deposit,10,A missed subscriber loses a deposit; a wasted call is cheap
4,gauravtopre/bank-customer-churn-dataset,churn,Customer leaves the bank,5,A lost customer costs lifetime value; a retention offer ...
5,blastchar/telco-customer-churn,Churn,Customer cancels service,5,As above
6,uciml/adult-census-income,income,Income above $50K,3,A missed high earner is a lost premium-product sale
7,sulianova/cardiovascular-disease-dataset,cardio,Has cardiovascular disease,5,A missed patient goes untreated; a false alarm costs a f...
8,jessemostipak/hotel-booking-demand,is_canceled,Booking is cancelled,3,An unexpected empty room costs a night's revenue; over-b...
9,pavansubhasht/ibm-hr-analytics-attrition-dataset,Attrition,Employee leaves,5,A lost employee costs rehiring; a retention talk is cheap


## 2. Helpers
Each helper returns numbers; the verdicts are set in the scorecard (section 4) with thresholds stated there.

In [3]:
def load(slug, pattern):
    root = kagglehub.dataset_download(slug)
    files = [os.path.join(dp, fn) for dp, _, fns in os.walk(root) for fn in fns
             if fn.lower().endswith(".csv") and re.search(pattern, fn, re.I)]
    f = max(files, key=os.path.getsize)
    with open(f, errors="ignore") as fh:
        head = fh.readline()
    return pd.read_csv(f, sep=";" if head.count(";") > head.count(",") else ",", low_memory=False), os.path.basename(f)

def to01(s, pos):
    if isinstance(pos, str) and pos.startswith(">="):
        return (pd.to_numeric(s, errors="coerce") >= float(pos[2:])).astype(int)
    if isinstance(pos, str):
        return (s.astype(str).str.strip().str.strip(".") == pos).astype(int)
    return (s == pos).astype(int)

def prepare(df, target, pos, drop):
    y = to01(df[target], pos)
    X = df.drop(columns=[target] + [c for c in drop if c in df.columns])
    for c in X.select_dtypes(exclude="number").columns:     # numbers stored as text
        v = pd.to_numeric(X[c].astype(str).str.strip().replace("", np.nan), errors="coerce")
        if v.notna().mean() > 0.95 and X[c].nunique() > 40:
            X[c] = v
    return X, y

def hygiene(df, X, y):
    txt = X.select_dtypes(exclude="number").columns
    token = sum(int(X[c].astype(str).str.strip().isin(MISSING_TOKENS).sum()) for c in txt)
    miss = (int(X.isna().sum().sum()) + token) / X.size
    feat_dup = X.duplicated()
    conflicts = int(pd.DataFrame({"h": pd.util.hash_pandas_object(X, index=False), "y": y.values}).groupby("h")["y"].nunique().gt(1).sum())
    return dict(rows=len(df), features=X.shape[1], text_features=len(txt), missing_pct=round(100 * miss, 2),
                duplicate_rows=int(df.duplicated().sum()), duplicate_pct=round(100 * df.duplicated().mean(), 2),
                conflicting_duplicates=conflicts, pos_rate_pct=round(100 * y.mean(), 2), val_positives=int(round(y.sum() * 0.2)))

def encode(X):
    txt = X.select_dtypes(exclude="number").columns
    Xe = pd.get_dummies(X, columns=[c for c in txt if X[c].nunique() <= 40], dtype=float)
    return Xe[[c for c in Xe.columns if pd.api.types.is_numeric_dtype(Xe[c]) or pd.api.types.is_bool_dtype(Xe[c])]].astype(float)

def ece(y, p, bins=10):
    y = np.asarray(y); p = np.clip(p, 0, 1); b = np.clip((p * bins).astype(int), 0, bins - 1)
    return float(sum((b == i).mean() * abs(y[b == i].mean() - p[b == i].mean()) for i in range(bins) if (b == i).any()))

def best_cost(y, p, ratio):
    y = np.asarray(y); ts = np.linspace(0.005, 0.995, 199)
    cost = np.array([(ratio * ((p < t) & (y == 1)).sum() + ((p >= t) & (y == 0)).sum()) / len(y) for t in ts])
    c05 = (ratio * ((p < 0.5) & (y == 1)).sum() + ((p >= 0.5) & (y == 0)).sum()) / len(y)
    return ts[cost.argmin()], cost.min(), c05

def modelling(Xe, y, ratio):
    dev, _ = train_test_split(np.arange(len(Xe)), test_size=0.2, stratify=y, random_state=SEED)
    if len(dev) > MAX_ROWS:
        dev = np.random.default_rng(0).choice(dev, MAX_ROWS, replace=False)
    Xd, yd = Xe.iloc[dev], y.iloc[dev].values
    single = {}
    for c in Xd.columns:
        v = Xd[c].fillna(Xd[c].median())
        if v.nunique() > 1:
            a = roc_auc_score(yd, v); single[c] = max(a, 1 - a)
    rank = pd.Series(single).sort_values(ascending=False)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    lin = lambda est: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), est)
    models = {"glass_L2": lin(Ridge()), "glass_log": lin(LogisticRegression(max_iter=3000)),
              "black_L2": HistGradientBoostingRegressor(random_state=SEED), "black_log": HistGradientBoostingClassifier(random_state=SEED)}
    S = {}
    for k, m in models.items():
        S[k] = cross_val_predict(m, Xd, yd, cv=cv, method="predict_proba")[:, 1] if k.endswith("log") else cross_val_predict(m, Xd, yd, cv=cv)
    folds = [te for _, te in cv.split(Xd, yd)]
    gaps = [roc_auc_score(yd[te], S["black_log"][te]) - roc_auc_score(yd[te], S["glass_log"][te]) for te in folds]
    t_star, c_star, c05 = best_cost(yd, S["black_log"], ratio)
    probe = cross_val_predict(HistGradientBoostingClassifier(random_state=SEED), Xd.drop(columns=[rank.index[0]]), yd, cv=cv, method="predict_proba")[:, 1]
    try:
        import shap
        m = HistGradientBoostingClassifier(random_state=SEED).fit(Xd, yd)
        sv = np.asarray(shap.TreeExplainer(m).shap_values(Xd.sample(min(2000, len(Xd)), random_state=SEED)))
        if sv.ndim == 3: sv = sv[..., 1]
        g = pd.Series(np.abs(sv).mean(0), index=Xd.columns).sort_values(ascending=False)
        shap_top, shap_share = g.index[0], g.iloc[0] / g.sum()
    except Exception as e:
        shap_top, shap_share = f"failed: {e!r}"[:40], np.nan
    auc = {k: roc_auc_score(yd, s) for k, s in S.items()}
    return dict(top_feature=rank.index[0], top_feature_auc=round(rank.iloc[0], 3),
                auc_glass_L2=round(auc["glass_L2"], 3), auc_glass_log=round(auc["glass_log"], 3),
                auc_black_L2=round(auc["black_L2"], 3), auc_black_log=round(auc["black_log"], 3),
                gap=round(np.mean(gaps), 3), gap_sd=round(np.std(gaps), 3),
                auc_without_top=round(roc_auc_score(yd, probe), 3),
                ece_black=round(ece(yd, S["black_log"]), 4),
                best_threshold=round(t_star, 3), cost_saving_pct=round(100 * (1 - c_star / c05), 1) if c05 else 0.0,
                shap_top=shap_top, shap_top_share=round(shap_share, 2))
print("helpers loaded")

helpers loaded


## 3. Run every candidate
A dataset that fails to download is recorded and skipped.

In [4]:
rows = []
for r in C.itertuples():
    t = time.time()
    try:
        df, fname = load(r.slug, r.pattern)
        df.columns = [c.strip() for c in df.columns]
        X, y = prepare(df, r.target, r.positive, r.drop)
        out = dict(dataset=r.slug, file=fname, **hygiene(df, X, y), **modelling(encode(X), y, r.cost_ratio))
    except Exception as e:
        out = dict(dataset=r.slug, error=repr(e)[:120])
    out["seconds"] = round(time.time() - t)
    rows.append(out)
    print(f"{r.slug:62s} {out.get('error', 'ok')}  ({out['seconds']}s)")
R = pd.DataFrame(rows).set_index("dataset")
R.to_csv("screening_results.csv")

laotse/credit-risk-dataset                                     ok  (12s)


uciml/default-of-credit-card-clients-dataset                   ok  (8s)


imakash3011/online-shoppers-purchasing-intention-dataset       ok  (8s)


henriqueyamahata/bank-marketing                                ok  (14s)


gauravtopre/bank-customer-churn-dataset                        ok  (6s)


blastchar/telco-customer-churn                                 ok  (8s)


... (7 outputs left out)

uciml/pima-indians-diabetes-database                           KaggleApiHTTPError("403 Client Error.\n\nYou don't have permission to access resource at URL: https://api.kaggle.com/v1/  (0s)


uciml/red-wine-quality-cortez-et-al-2009                       ok  (5s)


uom190346a/water-quality-and-potability                        ok  (6s)


uciml/faulty-steel-plates                                      ok  (7s)


In [5]:
display(R[["rows", "features", "missing_pct", "duplicate_pct", "conflicting_duplicates", "pos_rate_pct", "val_positives"]].dropna())
display(R[["top_feature", "top_feature_auc", "auc_glass_L2", "auc_glass_log", "auc_black_L2", "auc_black_log", "gap", "gap_sd",
           "auc_without_top", "ece_black", "best_threshold", "cost_saving_pct", "shap_top", "shap_top_share"]].dropna(subset=["auc_black_log"]))
if "error" in R: display(R.loc[R["error"].notna(), ["error"]])

,rows,features,missing_pct,duplicate_pct,conflicting_duplicates,pos_rate_pct,val_positives
dataset,,,,,,,
laotse/credit-risk-dataset,32581.0,11.0,1.12,0.51,0.0,21.82,1422.0
uciml/default-of-credit-card-clients-dataset,30000.0,23.0,0.00,0.00,21.0,22.12,1327.0
imakash3011/online-shoppers-purchasing-intention-dataset,12330.0,17.0,0.00,1.01,0.0,15.47,382.0
henriqueyamahata/bank-marketing,41188.0,19.0,1.63,0.03,237.0,11.27,928.0
gauravtopre/bank-customer-churn-dataset,10000.0,10.0,0.00,0.00,0.0,20.37,407.0
blastchar/telco-customer-churn,7043.0,19.0,0.01,0.00,18.0,26.54,374.0
uciml/adult-census-income,32561.0,14.0,0.93,0.07,1.0,24.08,1568.0
sulianova/cardiovascular-disease-dataset,70000.0,11.0,0.00,0.00,17.0,49.97,6996.0
jessemostipak/hotel-booking-demand,119390.0,27.0,0.02,26.80,267.0,37.04,8845.0


,top_feature,top_feature_auc,auc_glass_L2,auc_glass_log,auc_black_L2,auc_black_log,gap,gap_sd,auc_without_top,ece_black,best_threshold,cost_saving_pct,shap_top,shap_top_share
dataset,,,,,,,,,,,,,,
laotse/credit-risk-dataset,loan_percent_income,0.721,0.865,0.869,0.936,0.942,0.073,0.004,0.938,0.0071,0.170,14.6,loan_percent_income,0.17
uciml/default-of-credit-card-clients-dataset,PAY_0,0.693,0.721,0.726,0.782,0.782,0.056,0.009,0.752,0.0035,0.170,24.8,PAY_0,0.25
imakash3011/online-shoppers-purchasing-intention-dataset,PageValues,0.867,0.902,0.896,0.925,0.931,0.036,0.005,0.777,0.0180,0.135,31.0,PageValues,0.35
henriqueyamahata/bank-marketing,nr.employed,0.743,0.783,0.786,0.795,0.797,0.011,0.005,0.798,0.0052,0.100,38.3,nr.employed,0.27
gauravtopre/bank-customer-churn-dataset,age,0.728,0.765,0.762,0.853,0.858,0.095,0.010,0.802,0.0208,0.175,26.1,products_number,0.26
blastchar/telco-customer-churn,tenure,0.735,0.838,0.846,0.829,0.836,-0.009,0.007,0.836,0.0332,0.085,38.0,Contract_Month-to-month,0.19
uciml/adult-census-income,marital.status_Married-civ-spouse,0.760,0.893,0.907,0.924,0.927,0.020,0.001,0.926,0.0073,0.245,19.7,marital.status_Married-civ-spouse,0.23
sulianova/cardiovascular-disease-dataset,ap_hi,0.753,0.705,0.783,0.801,0.800,0.018,0.004,0.768,0.0045,0.165,46.3,ap_hi,0.48
jessemostipak/hotel-booking-demand,lead_time,0.686,0.842,0.859,0.900,0.903,0.044,0.001,0.892,0.0135,0.260,23.0,deposit_type_Non Refund,0.28


,error
dataset,
uciml/pima-indians-diabetes-database,"KaggleApiHTTPError(""403 Client Error.\n\nYou don't have ..."


## 4. Scorecard
✅ fine · ⚠️ manageable · ❌ conflicts with the brief. Thresholds:

| Check | ⚠️ | ❌ |
|---|---|---|
| Size | | more than 100,000 rows |
| Missing data | more than 1% of cells | more than 5% |
| Cleaning | more than 1% duplicate rows, or any duplicates with conflicting labels | more than 10% duplicate rows |
| Imbalance | positive rate under 10% | under 2% |
| Validation positives | under 100 | under 30 |
| Leakage | | one feature alone gives AUC ≥ 0.95 |
| Difficulty | best AUC ≥ 0.97 (plan to delete features) | best AUC ≤ 0.65 (too little signal) |
| Model story | gap under 2 × its fold-to-fold sd (the comparison may be noise) | |
| Calibration | blackbox ECE above 0.05 | |
| Threshold story | cost-minimising threshold saves under 10% vs 0.5 | |
| SHAP | top feature carries over 60% of the attribution | |

In [6]:
def flags(r):
    f = {}
    f["size"] = "bad" if r.rows > 100_000 else "ok"
    f["missing"] = "bad" if r.missing_pct > 5 else ("warn" if r.missing_pct > 1 else "ok")
    f["cleaning"] = "bad" if r.duplicate_pct > 10 else ("warn" if (r.duplicate_pct > 1 or r.conflicting_duplicates > 0) else "ok")
    f["imbalance"] = "bad" if r.pos_rate_pct < 2 else ("warn" if r.pos_rate_pct < 10 else "ok")
    f["val_positives"] = "bad" if r.val_positives < 30 else ("warn" if r.val_positives < 100 else "ok")
    f["leakage"] = "bad" if r.top_feature_auc >= 0.95 else "ok"
    best = max(r.auc_black_log, r.auc_glass_log)
    f["difficulty"] = "bad" if best <= 0.65 else ("warn" if best >= 0.97 else "ok")
    f["model_story"] = "warn" if abs(r.gap) < 2 * r.gap_sd else "ok"
    f["calibration"] = "warn" if r.ece_black > 0.05 else "ok"
    f["threshold"] = "warn" if r.cost_saving_pct < 10 else "ok"
    f["shap"] = "warn" if (r.shap_top_share > 0.6) else "ok"
    return pd.Series(f)

ok = R.dropna(subset=["auc_black_log"])
F = ok.apply(flags, axis=1)
summary = pd.DataFrame({"❌": (F == "bad").sum(1), "⚠️": (F == "warn").sum(1),
                        "blackbox AUC": ok.auc_black_log, "gap": ok.gap, "cost saving %": ok.cost_saving_pct}).sort_values(["❌", "⚠️", "gap"], ascending=[True, True, False])
icon = {"ok": "✅", "warn": "⚠️", "bad": "❌"}
display(F.loc[summary.index].replace(icon))
display(summary)

,size,missing,cleaning,imbalance,val_positives,leakage,difficulty,model_story,calibration,threshold,shap
dataset,,,,,,,,,,,
gauravtopre/bank-customer-churn-dataset,✅,✅,✅,✅,✅,✅,✅,✅,✅,✅,✅
laotse/credit-risk-dataset,✅,⚠️,✅,✅,✅,✅,✅,✅,✅,✅,✅
uciml/default-of-credit-card-clients-dataset,✅,✅,⚠️,✅,✅,✅,✅,✅,✅,✅,✅
imakash3011/online-shoppers-purchasing-intention-dataset,✅,✅,⚠️,✅,✅,✅,✅,✅,✅,✅,✅
uciml/adult-census-income,✅,✅,⚠️,✅,✅,✅,✅,✅,✅,✅,✅
sulianova/cardiovascular-disease-dataset,✅,✅,⚠️,✅,✅,✅,✅,✅,✅,✅,✅
uom190346a/water-quality-and-potability,✅,⚠️,✅,✅,✅,✅,✅,✅,⚠️,✅,✅
uciml/faulty-steel-plates,✅,✅,✅,✅,⚠️,✅,✅,✅,⚠️,✅,✅
henriqueyamahata/bank-marketing,✅,⚠️,⚠️,✅,✅,✅,✅,✅,✅,✅,✅


,❌,⚠️,blackbox AUC,gap,cost saving %
dataset,,,,,
gauravtopre/bank-customer-churn-dataset,0,0,0.858,0.095,26.1
laotse/credit-risk-dataset,0,1,0.942,0.073,14.6
uciml/default-of-credit-card-clients-dataset,0,1,0.782,0.056,24.8
imakash3011/online-shoppers-purchasing-intention-dataset,0,1,0.931,0.036,31.0
uciml/adult-census-income,0,1,0.927,0.020,19.7
sulianova/cardiovascular-disease-dataset,0,1,0.800,0.018,46.3
uom190346a/water-quality-and-potability,0,2,0.665,0.188,54.1
uciml/faulty-steel-plates,0,2,0.920,0.082,59.2
henriqueyamahata/bank-marketing,0,2,0.797,0.011,38.3


## 5. Ranking
Flag counts sort the datasets (section 4), but they don't capture everything the brief grades. The ranking also weighs: a stable gap between glassbox and blackbox (so the model decision means something), a cost story that comes from the context, how much the threshold step saves, real vs synthetic data, and copy risk. All numbers below are from the run above.

| Rank | Dataset | Evidence | Main risk |
|---|---|---|---|
| **1** | **Credit Risk** (`laotse/credit-risk-dataset`) | 32,581 loans, 21.8% default, about 1,420 validation positives. Glassbox 0.869 vs blackbox 0.942: a gap of +0.073 with a fold-to-fold sd of 0.004, so the comparison is stable. Not near-perfect, and robust: dropping the strongest feature leaves AUC at 0.938. Calibration error 0.007. SHAP spread out (top feature 17%). Clear cost story: a default loses principal, a rejected good borrower loses interest. | 1.1% missing (employment length, interest rate) and 0.5% duplicate rows. Likely simulated. The threshold saving at 5:1 is the smallest of the five (15%). |
| **2** | **Faulty Steel Plates, Bumps vs other faults** (`uciml/faulty-steel-plates`) | Real plant data, 1,941 plates, 20.7% Bumps, no missing values, no duplicates. Glassbox 0.838 vs blackbox 0.920: a gap of +0.082 with a fold-to-fold sd of 0.026. Robust: dropping the strongest feature leaves AUC at 0.920. SHAP spread out (top feature 10%). The largest threshold saving of any dataset: 59% at 10:1. Clear cost story: a faulty plate shipped triggers a claim, a false alarm costs a re-inspection. | Small: about 80 validation positives, so intervals are wide. Calibration error 0.061, the only ⚠️ on calibration among the top six. The rows are sorted by fault type and must be shuffled. |
| **3** | **Default of Credit Card Clients** (`uciml/default-of-credit-card-clients-dataset`) | Real bank data, 30,000 clients, 22.1% default, no missing values. Glassbox 0.726 vs blackbox 0.782 (+0.056, sd 0.009): the hardest of the five, so the model comparison has room to matter. Threshold saves 25%. Calibration error 0.004. | 21 groups of identical rows with conflicting labels. The repayment-status columns use codes (−2, −1, 0…) that need explaining. |
| **4** | **Online Shoppers Purchasing Intention** (`imakash3011/online-shoppers-purchasing-intention-dataset`) | Real sessions, 12,330 rows, 15.5% purchase, no missing values. Glassbox 0.896 vs blackbox 0.931 (+0.036, sd 0.005). Threshold saves 31%. A ready-made "delete a feature" step: dropping `PageValues` takes AUC from 0.931 to 0.777. | 1.0% duplicate rows. `PageValues` carries 35% of the SHAP attribution. |
| **5** | **Bank Customer Churn** (`gauravtopre/bank-customer-churn-dataset`) | The only dataset with no flags. 10,000 customers, 20.4% churn. The largest stable gap: glassbox 0.762 vs blackbox 0.858 (+0.095, sd 0.010). Threshold saves 26%. | Churn is the example used in the course's own slides, so the copy risk is highest and it does least to set the work apart. Likely synthetic. |
| **6** | **Bank Marketing** (`henriqueyamahata/bank-marketing`) | Real campaign data, 41,188 calls, 11.3% subscribe. Call `duration` is known only after the call, so it was removed: a leak-deletion step that is easy to explain. Threshold saves 38% at 10:1. | Small gap: 0.786 vs 0.797 (+0.011, sd 0.005), so the decision may well be "choose the glassbox". 1.6% "unknown" values and 237 groups of identical rows with conflicting labels. |

**Close but not in the top 5**
- **Adult income:** strong (0.907 vs 0.927) and large, but the target is income, and a cost story built on it raises fairness questions.
- **Cardiovascular disease:** 70,000 patients and a good screening cost story, but a small gap (+0.018) and 17 conflicting-duplicate groups.

**Not suitable**
- **Hotel bookings:** 119,390 rows, 26.8% duplicate rows.
- **Telco churn and HR attrition:** the glassbox beats the blackbox; HR has only about 47 validation positives.
- **Stroke:** 4.9% positive, about 50 validation positives.
- **Heart failure, German credit, red wine:** 900–1,600 rows. German credit has 6.4% missing; red wine has 15% duplicates.
- **Water potability:** the glassbox is no better than chance (0.476).
- **Pima diabetes:** could not be downloaded (Kaggle asks for a login).

**Still to check by hand:** the usability rating on each Kaggle page (links in the candidate table, section 1).